# Training Monitor

Interactive Plotly dashboard for monitoring PPO and expert iteration training.

Metrics are logged to `metrics/ppo_train.csv` and `metrics/exit_train.csv` during training.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

import plotly.io as pio
pio.templates.default = "plotly_white"

# Auto-detect project root (works from notebooks/ or project root)
HERE = Path(".").resolve()
PROJECT_ROOT = HERE if (HERE / "metrics").exists() else HERE.parent
METRICS_DIR = PROJECT_ROOT / "metrics"
print(f"Looking for metrics in: {METRICS_DIR}")

## Load Data

In [ ]:
ppo_path = METRICS_DIR / "ppo_train.csv"
exit_path = METRICS_DIR / "exit_train.csv"

ppo = pd.read_csv(ppo_path) if ppo_path.exists() else pd.DataFrame()
exit_df = pd.read_csv(exit_path) if exit_path.exists() else pd.DataFrame()

# Parse eval columns
if not ppo.empty:
    ppo["eval_win_rate"] = pd.to_numeric(ppo["eval_win_rate"], errors="coerce")
    ppo["eval_games"] = pd.to_numeric(ppo["eval_games"], errors="coerce")
    for c in ["pi_loss", "v_loss", "entropy", "clip_frac", "time_s"]:
        ppo[c] = pd.to_numeric(ppo[c], errors="coerce")
    ppo["win_rate"] = ppo["wins"] / (ppo["wins"] + ppo["losses"] + ppo["draws"])
    ppo["samples_per_sec"] = ppo["samples"] / ppo["time_s"]
    ppo["decisions_per_game"] = ppo["decisions"] / ppo["games"]

if not exit_df.empty:
    for c in ["pi_loss", "v_loss", "time_s"]:
        exit_df[c] = pd.to_numeric(exit_df[c], errors="coerce")

print(f"PPO rows: {len(ppo)}, Exit rows: {len(exit_df)}")
ppo.head() if not ppo.empty else print("No PPO metrics yet — run `just train` first.")

## Loss Curves (PPO)

Policy loss should hover near 0 (small positive/negative swings are normal).
Value loss should decrease over training.

In [ ]:
if not ppo.empty:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=["Policy Loss", "Value Loss"],
                        vertical_spacing=0.08)
    fig.add_trace(go.Scatter(x=ppo["iter"], y=ppo["pi_loss"], mode="lines+markers",
                             name="pi_loss", marker=dict(size=3)), row=1, col=1)
    fig.add_trace(go.Scatter(x=ppo["iter"], y=ppo["v_loss"], mode="lines+markers",
                             name="v_loss", marker=dict(size=3),
                             line=dict(color="orange")), row=2, col=1)
    fig.update_layout(height=500, title="PPO Loss Curves",
                      xaxis2_title="Iteration", showlegend=False)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=1)
    fig.show()

## Policy Health: Entropy & Clip Fraction

- **Entropy**: higher = more exploration. Should decrease slowly as policy becomes
  more confident. If it collapses to near 0, the policy is overfitting.
- **Clip fraction**: fraction of samples where PPO clipped the ratio. 1-10% is
  healthy; if it's consistently >20%, consider lowering the learning rate.

In [ ]:
if not ppo.empty:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=["Policy Entropy", "PPO Clip Fraction"],
                        vertical_spacing=0.08)
    fig.add_trace(go.Scatter(x=ppo["iter"], y=ppo["entropy"], mode="lines+markers",
                             name="entropy", marker=dict(size=3)), row=1, col=1)
    fig.add_trace(go.Scatter(x=ppo["iter"], y=ppo["clip_frac"], mode="lines+markers",
                             name="clip_frac", marker=dict(size=3),
                             line=dict(color="green")), row=2, col=1)
    fig.update_layout(height=450, title="Policy Health",
                      xaxis2_title="Iteration", showlegend=False)
    fig.show()

## Win Rate vs Random Agent

The key metric: how often does the trained policy beat a random agent?
50% = random play, 100% = perfect against random.

In [ ]:
if not ppo.empty and ppo["eval_win_rate"].notna().any():
    eval_rows = ppo.dropna(subset=["eval_win_rate"])
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=eval_rows["iter"], y=eval_rows["eval_win_rate"] * 100,
        mode="lines+markers", name="Win Rate vs Random",
        marker=dict(size=8), line=dict(width=2),
        hovertemplate="Iter %{x}<br>Win rate: %{y:.1f}%<extra></extra>"
    ))
    fig.add_hline(y=50, line_dash="dash", line_color="red", opacity=0.5,
                  annotation_text="Random baseline (50%)")
    fig.add_hline(y=80, line_dash="dot", line_color="green", opacity=0.3,
                  annotation_text="Strong (80%)")
    fig.update_layout(title="Eval: Win Rate vs Random Agent",
                      xaxis_title="Iteration", yaxis_title="Win Rate (%)",
                      yaxis_range=[0, 105], height=400)
    fig.show()
else:
    print("No eval data yet.")

## Self-Play Win/Loss Record

Win rate during self-play training games (not eval). Should hover around 50%
since the agent plays against itself or its past versions.

In [ ]:
if not ppo.empty:
    fig = go.Figure()
    fig.add_trace(go.Bar(x=ppo["iter"], y=ppo["wins"], name="Wins",
                         marker_color="#2ecc71"))
    fig.add_trace(go.Bar(x=ppo["iter"], y=ppo["losses"], name="Losses",
                         marker_color="#e74c3c"))
    if ppo["draws"].sum() > 0:
        fig.add_trace(go.Bar(x=ppo["iter"], y=ppo["draws"], name="Draws",
                             marker_color="#95a5a6"))
    fig.update_layout(barmode="stack", title="Self-Play Game Outcomes",
                      xaxis_title="Iteration", yaxis_title="Games",
                      height=350)
    fig.show()

## Training Throughput

How many samples per second and decisions per game. Useful for spotting
performance regressions or understanding iteration time.

In [ ]:
if not ppo.empty:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=["Samples/sec", "Decisions per Game"],
                        vertical_spacing=0.08)
    fig.add_trace(go.Scatter(x=ppo["iter"], y=ppo["samples_per_sec"],
                             mode="lines+markers", name="samples/s",
                             marker=dict(size=3)), row=1, col=1)
    fig.add_trace(go.Scatter(x=ppo["iter"], y=ppo["decisions_per_game"],
                             mode="lines+markers", name="decisions/game",
                             marker=dict(size=3),
                             line=dict(color="purple")), row=2, col=1)
    fig.update_layout(height=450, title="Training Throughput",
                      xaxis2_title="Iteration", showlegend=False)
    fig.show()

## Expert Iteration (Phase 2)

If you've run `just exit-train`, this shows the MCTS-based training metrics.

In [ ]:
if not exit_df.empty:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=["Exit Policy Loss", "Exit Value Loss"],
                        vertical_spacing=0.08)
    fig.add_trace(go.Scatter(x=exit_df["iter"], y=exit_df["pi_loss"],
                             mode="lines+markers", name="pi_loss",
                             marker=dict(size=6)), row=1, col=1)
    fig.add_trace(go.Scatter(x=exit_df["iter"], y=exit_df["v_loss"],
                             mode="lines+markers", name="v_loss",
                             marker=dict(size=6),
                             line=dict(color="orange")), row=2, col=1)
    fig.update_layout(height=450, title="Expert Iteration Loss Curves",
                      xaxis2_title="Iteration", showlegend=False)
    fig.show()
else:
    print("No exit training metrics yet — run `just exit-train` first.")

## Summary Stats

In [ ]:
if not ppo.empty:
    last = ppo.iloc[-1]
    evals = ppo.dropna(subset=["eval_win_rate"])
    best_eval = evals.loc[evals["eval_win_rate"].idxmax()] if not evals.empty else None
    
    print("=== PPO Training Summary ===")
    print(f"Iterations:     {int(last['iter'])}")
    print(f"Last pi_loss:   {last['pi_loss']:.4f}")
    print(f"Last v_loss:    {last['v_loss']:.4f}")
    print(f"Last entropy:   {last['entropy']:.3f}")
    print(f"Total time:     {ppo['time_s'].sum():.0f}s ({ppo['time_s'].sum()/60:.1f}m)")
    if best_eval is not None:
        print(f"Best eval:      {best_eval['eval_win_rate']:.1%} at iter {int(best_eval['iter'])}")
    print(f"Latest eval:    {last['eval_win_rate']:.1%}" if pd.notna(last['eval_win_rate']) else "")

if not exit_df.empty:
    print(f"\n=== Exit Training ===")
    print(f"Iterations:     {len(exit_df)}")
    print(f"Last pi_loss:   {exit_df.iloc[-1]['pi_loss']:.4f}")
    print(f"Last v_loss:    {exit_df.iloc[-1]['v_loss']:.4f}")